In [ ]:
#| default_exp hx_bridge

In [ ]:
# await greet('dutil.hx_bridge')

# htmx API python bridge
> Using `htmx` javascript API from python

See exploration <a href="/dialog_?name=vic%2Fdutil%2Fexplorer%2Fhtmx_bridge" target="_blank">vic/dutil/explorer/htmx_bridge</a>

<!-- linkedto: _21a088dd -->
Solveit version: **0.0.85**  
dialoghelper version: **0.2.16**  
git branch: **dev-log**  
git changes: [' M ../dutil/hx_bridge.js', ' M ../explorer/htmx_bridge.ipynb', ' M 00_htmx_bridge.ipynb', '?? ../explorer/.sesskey']

Hi! I'm Vic, we're going to work together on the project **dutil.hx_bridge**.

In [ ]:
#| export
from importlib import resources
from uuid import uuid4
from fastcore.meta import delegates
from fastcore.xml import to_xml
from dialoghelper.core import iife_a, event_get_a, allow, read_msg, find_dname
from dutil.server import local_server, wrap_endp

In [ ]:
import asyncio, inspect, subprocess, json, pydoc
from contextlib import asynccontextmanager
from uuid import uuid4
from IPython.display import HTML, Markdown
import fastcore.all as FC
from fastcore.foundation import AttrDict as AD
from fastcore.meta import delegates
from fastcore.test import *
from fastcore.xml import FT
from httpx import get as xget, post as xpost
from starlette.routing import Route
from starlette.responses import HTMLResponse
from fasthtml.common import *
from fasthtml.components import *
from fasthtml.jupyter import *
from dialoghelper.core import doc
from dutil.core import link_msg, ctxusage
from dutil.server import find_server, local_server

## Goal

Implement a syncronous python bridge to `htmx` API using current solveit capabilities that dialogs can use to interact with the browser. It will be modeled after the solveit tool `capture_screen`.

### scope

The goal of `htmx_bridge` project is to achieve parity between the JavaScript and Python htmx APIs, or as much as possible, and to document what is not possible or behaves differently.

Looking at the htmx JS API reference in the dialog, the methods fall into a few natural categories:

**DOM manipulation** — `find`, `findAll`, `closest`, `remove`, `addClass`, `removeClass`, `toggleClass`, `takeClass` — these return or operate on DOM elements. Not serializable directly, but we could return `outerHTML` or just a success/error.

**Behavior** — `ajax`, `swap`, `process`, `trigger`, `on`, `off` — side effects, mostly fire-and-forget or Promise-based.

**Utility** — `values`, `parseInterval`, `logAll`, `logNone` — return plain serializable values, easy to wrap.

**Config** — `htmx.config` is just a property, readable/writable via a dedicated `get_config`/`set_config` pair.

**Property** - `createEventSource`, `createWebSockect`, `logger`

**Other** - `defineExtension`, `removeExtension`, `onLoad`

For automation, the key insight is that **most wrappers are identical boilerplate** — just `_htmx_call('methodName', *args, **kwargs)`.

## JS bridge

Set up the JS leg of the bridge:

In [ ]:
js_bridge = (await read_msg()).content.replace('%%js\n', '')
js_path = resources.files('dutil')/'hx_bridge.js'
js_path.write_text(js_bridge)

8163

In [ ]:
#| export
async def setup_htmx_bridge(debugger:bool=False):
    "Inject htmx bridge into the browser"
    txt = (resources.files('dutil')/'hx_bridge.js').read_text()
    await iife_a(f"{'debugger;\n' if debugger else ''}"+txt)

## python API

In [ ]:
#| export
async def _htmx_call(method, *args, port=8000, full_response=False, timeout=10, _retry=True):
    "Low-level async call to htmx JS API"
    res = await event_get_a('hx_bridge', timeout=timeout, data=dict(method=method, args=args, port=port, full_response=full_response))
    try: del res['data_id']
    except Exception:
        if _retry:
            await setup_htmx_bridge()
            return await _htmx_call(method, *args, port=port, full_response=full_response, timeout=timeout, _retry=False)
    return res

In [ ]:
#| export
@delegates(_htmx_call)
async def js_eval(expr, timeout=2, **kwargs):
    "Evaluate a JS expression in the browser and return the result"
    return await _htmx_call('eval', expr, timeout=timeout, **kwargs)

In [ ]:
res = await js_eval('navigator.userAgent')
test_is('success' in res, True)
res

```python
{ 'success': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
             'AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 '
             'Safari/537.36 Edg/145.0.0.0'}
```

### Method - htmx.find()

Finds an element matching the selector

##### Parameters

* `selector` - the selector to match

or

* `elt` - the root element to find the matching element in, inclusive
* `selector` - the selector to match

##### Example

```js
    // find div with id my-div
    var div = htmx.find("#my-div")

    // find div with id another-div within that div
    var anotherDiv = htmx.find(div, "#another-div")
```

In [ ]:
# await setup_htmx_bridge()#debugger=True)

In [ ]:
#| export
@delegates(_htmx_call)
async def find(selector, root=None, **kwargs):
    "Find an element matching selector, optionally within root"
    return await _htmx_call('find', *([root, selector] if root else [selector]), **kwargs)

In [ ]:
dom_test = Div(
    P('paragraph one', id='p1'),
    P('paragraph two', id='p2'),
    Span('a span', id='sp1'),
    id='dom_root')
dom_test

<div id="dom_root"><p id="p1">paragraph one</p><p id="p2">paragraph two</p><span id="sp1">a span</span></div>

In [ ]:
res = await find('#p1')
test_eq(res.success.outerHTML, '<p id="p1">paragraph one</p>')
res

```python
{ 'success': { 'id': 'p1',
               'outerHTML': '<p id="p1">paragraph one</p>',
               'tagName': 'p'}}
```

In [ ]:
res = await find('#sp1', '#dom_root')
test_eq(res.success.outerHTML, '<span id="sp1">a span</span>')
res

```python
{ 'success': { 'id': 'sp1',
               'outerHTML': '<span id="sp1">a span</span>',
               'tagName': 'span'}}
```

In [ ]:
res = await find('#does-not-exist')
test_is('error' in res, True)
res

```python
{'error': 'No element found: #does-not-exist'}
```

In [ ]:
res = await find('!!!invalid')
test_is('error' in res, True)
test_is('SyntaxError' in res.error, True)
res

```python
{ 'error': "SyntaxError: Failed to execute 'querySelector' on 'Document': "
           "'!!!invalid' is not a valid selector."}
```

In [ ]:
res = await find('h1')
test_eq(res.success.tagName, 'h1')
res

```python
{ 'success': { 'id': '',
               'outerHTML': '<h1 uk-tooltip="Close dialog" class="uk-h1 '
                            'text-nowrap text-2xl md:text-3xl pr-1.5" '
                            'tabindex="0">📚 solveit</h1>',
               'tagName': 'h1'}}
```

In [ ]:
res = await find('#sp1', '#nonexistent-root')
test_is('error' in res, True)
res

```python
{'error': 'Error: Root not found: #nonexistent-root'}
```

In [ ]:
res = await find('p')
test_eq(res.success.tagName, 'p')
res

```python
{ 'success': { 'id': '',
               'outerHTML': '<p class="text-gray-500 dark:text-gray-200 '
                            'text-sm">\n'
                            'Code: 40<span class="time-el"> (<span '
                            'id="time-96247b17">10:12:00 '
                            'AM</span><script>document.getElementById("time-96247b17").textContent '
                            '= new '
                            'Date("2026-03-11T09:12:00.508796+00:00").toLocaleString("en-US", '
                            '{hour:"2-digit", minute:"2-digit", '
                            'second:"2-digit"})</script>)</span>          </p>',
               'tagName': 'p'}}
```

In [ ]:
res = await find('')
test_is('error' in res, True)
res

```python
{ 'error': "SyntaxError: Failed to execute 'querySelector' on 'Document': The "
           'provided selector is empty.'}
```

### Method - htmx.findAll()

Finds all elements matching the selector

##### Parameters

* `selector` - the selector to match

or

* `elt` - the root element to find the matching elements in, inclusive
* `selector` - the selector to match

##### Example

```js
    // find all divs
    var allDivs = htmx.findAll("div")

    // find all paragraphs within a given div
    var allParagraphsInMyDiv = htmx.findAll(htmx.find("#my-div"), "p")
```

In [ ]:
#| export
@delegates(_htmx_call)
async def findAll(selector, root=None, **kwargs):
    "Find all elements matching `selector`, optionally within `root`"
    return await _htmx_call('findAll', *([root, selector] if root else [selector]), **kwargs)

In [ ]:
res = await findAll('p')
res.success.count

267

In [ ]:
res = await findAll('.no-such-class')
test_eq(res.success.count, 0)
res

```python
{'success': {'count': 0, 'items': []}}
```

In [ ]:
# findAll with root
res = await findAll('p', '#dom_root')
test_eq(res.success.count, 2)
res

```python
{ 'success': { 'count': 2,
               'items': [{'id': 'p1', 'tagName': 'p', 'outerHTML': '<p id="p1">paragraph one</p>'}, {'id': 'p2', 'tagName': 'p', 'outerHTML': '<p id="p2">paragraph two</p>'}]}}
```

### Method - htmx.swap()

Performs swapping (and settling) of HTML content

##### Parameters

* `target` - the HTML element or string selector of swap target
* `content` - string representation of content to be swapped
* `swapSpec` - swapping specification, representing parameters from `hx-swap`
  * `swapStyle` (required) - swapping style (`innerHTML`, `outerHTML`, `beforebegin` etc)
  * `swapDelay`, `settleDelay` (number) - delays before swapping and settling respectively
  * `transition` (bool) - whether to use HTML transitions for swap
  * `ignoreTitle` (bool) - disables page title updates
  * `head` (string) - specifies `head` tag handling strategy (`merge` or `append`). Leave empty to disable head handling
  * `scroll`, `scrollTarget`, `show`, `showTarget`, `focusScroll` - specifies scroll handling after swap
* `swapOptions` - additional *optional* parameters for swapping
  * `select` - selector for the content to be swapped (equivalent of `hx-select`)
  * `selectOOB` - selector for the content to be swapped out-of-band (equivalent of `hx-select-oob`)
  * `eventInfo` - an object to be attached to `htmx:afterSwap` and `htmx:afterSettle` elements
  * `anchor` - an anchor element that triggered scroll, will be scrolled into view on settle. Provides simple alternative to full scroll handling
  * `contextElement` - DOM element that serves as context to swapping operation. Currently used to find extensions enabled for specific element
  * `afterSwapCallback`, `afterSettleCallback` - callback functions called after swap and settle respectively. Take no arguments


##### Example

```js
    // swap #output element inner HTML with div element with "Swapped!" text
    htmx.swap("#output", "<div>Swapped!</div>", {swapStyle: 'innerHTML'});
```

In [ ]:
#| export
@delegates(_htmx_call)
async def swap(target, content, swapSpec=None, swapOptions=None, **kwargs):
    "Perform an htmx swap, blocking until complete"
    if not isinstance(content, str): content = to_xml(content)
    spec = dict(swapStyle='innerHTML', scroll=False, show=False) | (swapSpec or {})
    return await _htmx_call('swap', target, content, spec, swapOptions or {}, **kwargs)

#### Simple test

`swap` needs no server. We just need a target element in the page and swap some content into it.

In [ ]:
fh_cfg['auto_id']=True

In [ ]:
test_ft = Div('original content')
test_ft

<div id="_rgBTJfLqTlyGUsYm0kDeVw">original content</div>

In [ ]:
str(test_ft) # <-- idiomatic fasthtml to get an element id

'_rgBTJfLqTlyGUsYm0kDeVw'

In [ ]:
res = await swap(f"#{test_ft}", P('hello from python!'))
display(res)
print('done!')
test_eq(res.success, {'swapped': True})
test_is('hello from python!' in (await find(f"#{test_ft}")).success.outerHTML, True)

```python
{'success': {'swapped': True}}
```

done!


Let's set up a test playground with a few target elements, then systematically test each `swapSpec` and `swapOptions` feature:

</details>

First, render the target elements we'll use across all tests:

In [ ]:
targets = Ul(
    Li(Pre('1. swapStyle: innerHTML (default)'), Div('target: innerHTML', id='t-inner')),
    Li(Pre('2. swapStyle: outerHTML'), Div('target: outerHTML', id='t-outer')),
    Li(Pre('3. swapStyle: beforebegin / afterend'), Div('target: beforebegin', id='t-before')),
    Li(Pre('3. swapStyle: beforebegin / afterend'), Div('target: afterend', id='t-after')),
    Li(Pre('4. swapStyle: delete (ignores content)'), Div('target: delete', id='t-delete')),
    Li(Pre('5. swapDelay + settleDelay'), Div(Div('child 1'), Div('child 2'), id='t-children')),
    Li(Pre('6. selectOOB - swap an additional element out of band'), Div(id='t-oob')),
    Li(Pre(''), Div('target: transition', id='t-transition')),
    style='display:flex;flex-direction:column;gap:8px;padding:8px;border:1px solid #ccc'
)
targets

<ul id="_ocac3y_UTYabYp69XAo7hg" style="display:flex;flex-direction:column;gap:8px;padding:8px;border:1px solid #ccc"><li id="_mzV2ACgZQlGnCN79n6Snkg"><pre id="_sePY0zc8SCS32iadrRCudQ">1. swapStyle: innerHTML (default)</pre><div id="t-inner">target: innerHTML</div></li><li id="_Lj6_eRJ5R1KeGimoZkKfrw"><pre id="_AXYxV8gcSiOrCsUHRCDzXA">2. swapStyle: outerHTML</pre><div id="t-outer">target: outerHTML</div></li><li id="_TdvhGtNpQ1ymTNkoJyUSGA"><pre id="_283zOWbCSMGzJziqEnf2sA">3. swapStyle: beforebegin / afterend</pre><div id="t-before">target: beforebegin</div></li><li id="_Om0euELHR32j_GWKxGJ97Q"><pre id="_v32y6UhHRVui2ud3cUmCWA">3. swapStyle: beforebegin / afterend</pre><div id="t-after">target: afterend</div></li><li id="_HiB9VfMwQNOm7GPbUrYC_A"><pre id="_My1I0R4iSeOkFQQrzd0xRw">4. swapStyle: delete (ignores content)</pre><div id="t-delete">target: delete</div></li><li id="_GW1jP7bZRHq8Iu9PO46q6A"><pre id="_7tpTpqShTN_ozHkDwzdYeQ">5. swapDelay + settleDelay</pre><div id="t-children"><div id="_nRQUOXlsSmOL0bkv8XUWlg">child 1</div><div id="_RBoZ0-JOQ1u0cWaiK2OUAg">child 2</div></div></li><li id="_V7yts7luTI6fC_cV_c2sPA"><pre id="_IZ8DCWzFS0y99vHoW9PcEw">6. selectOOB - swap an additional element out of band</pre><div id="t-oob"></div></li><li id="_HBB3P4iTSb2bHvpXeyE7Bw"><pre id="_bfW7le0PQ_eBRROKvltfIQ"></pre><div id="t-transition">target: transition</div></li></ul>

In [ ]:
# 1. swapStyle: innerHTML (default)
res = await swap('#t-inner', Div('✅ innerHTML swap'))
test_eq(res.success, {'swapped': True})

In [ ]:
# 2. swapStyle: outerHTML
res = await swap('#t-outer', Div('✅ outerHTML swap', id='t-outer'), swapSpec=dict(swapStyle='outerHTML'))
test_eq(res.success, {'swapped': True})

In [ ]:
# 3. swapStyle: beforebegin / afterend
res = await swap('#t-before', Div('✅ beforebegin'), swapSpec=dict(swapStyle='beforebegin'))
test_eq(res.success, {'swapped': True})
res = await swap('#t-after', Div('✅ afterend'), swapSpec=dict(swapStyle='afterend'))
test_eq(res.success, {'swapped': True})

In [ ]:
# 4. swapStyle: delete (ignores content)
res = await swap('#t-delete', '', swapSpec=dict(swapStyle='delete'))
test_eq(res.success, {'swapped': True})

In [ ]:
# 5. swapDelay + settleDelay
res = await swap('#t-children', Div('✅ delayed swap'), swapSpec=dict(swapStyle='innerHTML', swapDelay=200, settleDelay=100))
test_eq(res.success, {'swapped': True})

In [ ]:
# 6. selectOOB - swap an additional element out of band
res = await swap('#t-inner', f'<div>main content</div><div id="t-oob" hx-swap-oob="true">✅ OOB swap</div>', swapOptions=dict(selectOOB='#t-oob'))
test_eq(res.success, {'swapped': True})

In [ ]:
# 7. select (hx-select equivalent - pick part of response)
res = await swap('#t-inner', '<div><p id="pick-me">✅ selected</p><p>ignored</p></div>', swapOptions=dict(select='#pick-me'))
test_eq(res.success, {'swapped': True})

In [ ]:
# 8. transition (View Transitions API - may not be supported in all browsers)
res = await swap('#t-transition', Div('✅ transition swap'), swapSpec=dict(swapStyle='innerHTML', transition=True))
test_eq(res.success, {'swapped': True})  # expected: success or error depending on browser support

In [ ]:
results = {}

# ignoreTitle
r = await swap('#t-inner', '<title>should be ignored</title><div>✅ ignoreTitle</div>', swapSpec=dict(swapStyle='innerHTML', ignoreTitle=True))
results['ignoreTitle'] = r

# head:merge
r = await swap('#t-inner', '<head><meta name="test" content="val"></head><div>✅ head merge</div>', swapSpec=dict(swapStyle='innerHTML', head='merge'))
results['head:merge'] = r

# scroll:top
r = await swap('#t-inner', '<div>✅ scroll:top</div>', swapSpec=dict(swapStyle='innerHTML', scroll='top'))
results['scroll:top'] = r

# show:top
r = await swap('#t-inner', '<div>✅ show:top</div>', swapSpec=dict(swapStyle='innerHTML', show='top'))
results['show:top'] = r

# focusScroll
r = await swap('#t-inner', '<div>✅ focusScroll</div>', swapSpec=dict(swapStyle='innerHTML', focusScroll=True))
results['focusScroll'] = r

# scrollTarget
r = await swap('#t-inner', '<div>✅ scrollTarget</div>', swapSpec=dict(swapStyle='innerHTML', scrollTarget='#t-outer'))
results['scrollTarget'] = r

# showTarget
r = await swap('#t-inner', '<div>✅ showTarget</div>', swapSpec=dict(swapStyle='innerHTML', showTarget='#t-outer'))
results['showTarget'] = r

results

{'ignoreTitle': {'success': {'swapped': True}},
 'head:merge': {'success': {'swapped': True}},
 'scroll:top': {'success': {'swapped': True}},
 'show:top': {'success': {'swapped': True}},
 'focusScroll': {'success': {'swapped': True}},
 'scrollTarget': {'success': {'swapped': True}},
 'showTarget': {'success': {'swapped': True}}}

There are a few distinct error cases worth testing: invalid selector syntax, non-existent target, invalid `swapStyle`, and the async path (with `swapDelay`) where the current `try/catch` in the JS handler won't catch a re-thrown error since it escapes via `setTimeout`.

In [ ]:
# sync: invalid selector syntax → JS SyntaxError, caught by try/catch
res = await swap('!!!bad', Div('x'))
test_eq('SyntaxError' in res.error, True)
res

```python
{ 'error': "SyntaxError: Failed to execute 'querySelector' on 'Document': "
           "'!!!bad' is not a valid selector."}
```

In [ ]:
# sync: non-existent target → htmx re-throws after firing htmx:swapError, caught
res = await swap('#nonexistent-target', Div('x'))
test_eq('TypeError' in res.error, True)
res

```python
{'error': "TypeError: Cannot read properties of null (reading 'firstChild')"}
```

In [ ]:
# sync: invalid swapStyle → htmx silently no-ops (no error thrown)
res = await swap('#t-inner', Div('x'), swapSpec=dict(swapStyle='bogus'))
test_eq(res.success, {'swapped': True})
print('invalid swapStyle:', res)

invalid swapStyle: {'success': {'swapped': True}}


In [ ]:
res = await swap('#nonexistent-target', Div('x'), swapSpec=dict(swapStyle='innerHTML', swapDelay=100), timeout=5)
test_eq('Target not found' in res.error, True)
res

```python
{'error': 'Error: Target not found: #nonexistent-target'}
```

#### swap() — findings & caveats

**What works from Python (all `swapSpec` fields):**
- `swapStyle` — all styles: `innerHTML`, `outerHTML`, `beforebegin`, `afterbegin`, `beforeend`, `afterend`, `delete`, `textContent`
- `swapDelay`, `settleDelay` — fully async-safe: bridge uses a one-shot `htmx:afterSettle` listener discriminated by `idx` via `eventInfo`
- `ignoreTitle`, `head` (`merge`/`append`) — work as expected
- `scroll`, `show`, `scrollTarget`, `showTarget`, `focusScroll` — work; default is `scroll=False, show=False` to suppress htmx's auto-scroll behavior in programmatic use
- `transition` — works if View Transitions API is available in the browser; async path handled correctly

**What works from Python (`swapOptions`):**
- `select` — picks a sub-element from content before swapping (equivalent of `hx-select`)
- `selectOOB` — swaps additional out-of-band elements; pass as CSS selector string (e.g. `'#my-oob'`)
- `anchor` — pass the element **id without `#`** (htmx resolves it internally as `'#' + anchor`)
- `eventInfo` — any JSON-serializable dict; attached to `htmx:afterSwap` and `htmx:afterSettle` detail. The bridge injects `{idx: ...}` here for async discrimination — **do not overwrite `idx`** if passing custom `eventInfo`

**Not applicable from Python:**
- `contextElement` — requires a live DOM element reference; N/A. Extensions scoped to specific elements won't be picked up
- `afterSwapCallback` / `afterSettleCallback` — JS functions, not serializable. Not needed: both fire **inside** `htmx.swap()` before it returns (synchronous path), so sequential Python code after `await swap(...)` is the exact equivalent
- `beforeSwapCallback` — same as above

**Timing guarantees:**
- Synchronous path (no `swapDelay`, `settleDelay`, `transition`): `await swap(...)` returns after full settle — DOM is stable
- Async path (`swapDelay > 0`, `settleDelay > 0`, `transition=True`): bridge waits for `htmx:afterSettle` — same guarantee
- Safe to call `find()`/`js_eval()` immediately after `await swap(...)` in all cases

**`class=""` artefact:** htmx leaves an empty `class` attribute on settled elements — harmless, expected behavior.

### Method - htmx.ajax()

Issues an htmx-style AJAX request. This method returns a Promise, so a callback can be executed after the content has been inserted into the DOM.

##### Parameters

* `verb` - 'GET', 'POST', etc.
* `path` - the URL path to make the AJAX
* `element` - the element to target (defaults to the `body`)

or

* `verb` - 'GET', 'POST', etc.
* `path` - the URL path to make the AJAX
* `selector` - a selector for the target

or

* `verb` - 'GET', 'POST', etc.
* `path` - the URL path to make the AJAX
* `context` - a context object that contains any of the following
    * `source` - the source element of the request, `hx-*` attrs which affect the request will be resolved against that element and its ancestors
    * `event` - an event that "triggered" the request
    * `handler` - a callback that will handle the response HTML
    * `target` - the target to swap the response into
    * `swap` - how the response will be swapped in relative to the target
    * `values` - values to submit with the request
    * `headers` - headers to submit with the request
    * `select` - allows you to select the content you want swapped from a response
    * `selectOOB` - allows you to select content for out-of-band swaps from a response
    * `push` - can be `'true'` or a path to push a URL into browser location history
    * `replace` - can be `'true'` or a path to replace the URL in the browser location history

##### Example

```js
    // issue a GET to /example and put the response HTML into #myDiv
    htmx.ajax('GET', '/example', '#myDiv')

    // issue a GET to /example and replace #myDiv with the response
    htmx.ajax('GET', '/example', {target:'#myDiv', swap:'outerHTML'})

    // execute some code after the content has been inserted into the DOM
    htmx.ajax('GET', '/example', '#myDiv').then(() => {
      // this code will be executed after the 'htmx:afterOnLoad' event,
      // and before the 'htmx:xhr:loadend' event
      console.log('Content inserted successfully!');
    });

```

In [ ]:
@delegates(_htmx_call)
async def ajax(verb:str, path:str, context:str|dict=None, **kwargs):
    "Issue an htmx-style AJAX request, blocking until complete"
    return await _htmx_call('ajax', verb, path, context, **kwargs)

Minimal `ajax` test rig — a FastHTML app with one route, served via `JupyUvi`:

In [ ]:
app, rt = fast_app()

@rt('/example')
def get(): return P('server response!')

server = JupyUvi(app)#, log_level='trace')

ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use


KeyboardInterrupt: 

In [ ]:
server.server.started
# server.stop()

In [ ]:
target = Div('waiting...', id='myDiv')
target

In [ ]:
res = await ajax('GET', '/example', '#myDiv')
display(res)
test_is(res.success, True)

`ajax` is a special case because is very low-level and can do a lot of different things depending on the context argument. I'm going to use this initially to test connection python-browser in other projects.

In [ ]:
res = await ajax('GET', '/example', '#myDiv', full_response=True)
test_is('response' in res, True)
test_is('xhr' in res.response, True)
res

For `swap` the `response` key will have settle detail (no `xhr`, but `successful`/`failed` will be absent too since settle detail differs from request detail — `_ser_detail` will just return `{}`). Worth seeing the raw output to decide if `swap`'s `full_response` is useful enough to keep or document as N/A.

In [ ]:
# make swap
res = await swap('#myDiv', swpd := P('full_response test'), full_response=True)
test_eq(res.success, {'swapped': True})
test_eq((await find(f"#{swpd}")).success.id, str(swpd))
res

**`full_response` caveat:** only meaningful for `ajax` — `find` and `swap` have no request detail to serialize, so the flag is silently ignored and the return dict is unchanged.

In [ ]:
@rt
def hello(): return P('hello!', id='p3')

@rt
def greet(name:str): return P(f'hello {name}!')

@rt('/headers-echo')
def get(req): return P(req.headers.get('X-Custom', 'none'))

@rt('/not-found')
def get(): return Response('oops', status_code=404)

In [ ]:
target = Div('waiting...', id='myDiv2')
target

In [ ]:
# 1. GET, string selector target
res = await ajax('GET', '/hello', '#myDiv2')
test_eq(res.success, True)
display(res)

res = (await find('#myDiv2')).success
test_is('hello!' in res.outerHTML, True)

In [ ]:
# 2. GET, context dict with swap
res = await ajax('GET', '/hello', dict(target='#myDiv2', swap='outerHTML'))
test_eq(res.success, True)
display(res)

res = (await find('#p3')).success
test_is('hello!' in res.outerHTML, True)
res

In [ ]:
# 3. POST with values
res = await ajax('POST', '/greet', dict(target='#myDiv', values=dict(name='world')))
test_eq(res.success, True)
res

In [ ]:
# POST with values
res = await ajax('POST', '/greet', dict(target='#myDiv', values=dict(name='world')), full_response=True)
test_eq(res.success, True)
test_is('hello world!' in res.response.xhr.responseText, True)
res

In [ ]:
target = Div('waiting...', id='myDiv3')
target

In [ ]:
# 4. custom headers
res = await ajax('GET', '/headers-echo', dict(target='#myDiv3', headers={'X-Custom': 'bridge-test'}), full_response=True)
test_eq(res.success, True)
res

In [ ]:
# 5. 404 — should come back as error
res = await ajax('GET', '/not-found', '#myDiv3')
test_is('error' in res, True)
res

In [ ]:
# 6. bad target selector — htmx aborts silently via DUMMY_ELT; check what we get
res = await ajax('GET', '/hello', '#does-not-exist')
test_is('targetError' in res.error, True)
res

In [ ]:
res = await ajax('GET', '/hello', '#does-not-exist', full_response=True)
test_is('targetError' in res.error, True)
res

In [ ]:
# 7. no context - no swap (swapstyle = 'none') but with the rest of htmx handling
res = await ajax('GET', '/hello')
test_eq(res.success, True)
res

In [ ]:
res = await ajax('GET', '/hello', full_response=True)
test_eq(res.success, True)
test_eq(res.response.xhr.responseText.strip(), to_xml(hello()).strip())
res

In [ ]:
# 8. full_response on success
res = await ajax('GET', '/hello', '#myDiv3', full_response=True)
test_eq(res.response.xhr.status, 200)
res

In [ ]:
# 9. full_response on 404
res = await ajax('GET', '/not-found', '#myDiv3', full_response=True)
test_is('error' in res, True)
res

What `ajax` actually adds over just using `httpx` directly from Python.

`httpx` gives full request/response introspection (headers, status, body), easy to test FastHTML/FastAPI routes, and no browser involvement needed.

`htmx.ajax` uniquely adds requests that goes through the **browser** (so cookies, CORS, browser auth, and session state are all real). `htmx` processes the response — the swap actually happens in the live DOM, so we can verify the full round-trip including the rendered result. `htmx` response headers are honoured — `HX-Trigger`, `HX-Push-Url`, `HX-Redirect`, `HX-Reswap` etc.

So the niche is specifically: **"does this htmx-aware endpoint behave correctly end-to-end in a real browser context?"** — not just "does it return the right HTML?", but "does it trigger the right events, swap into the right target, update the URL correctly?"

In [ ]:
@rt 
def ok(): return P('ok!')

@rt
def created(): return Response('created', status_code=201)

@rt
async def slow(): await asyncio.sleep(0.5); return P('slow!')

In [ ]:
r1, r2 = await asyncio.gather(ajax('GET', '/slow', full_response=True), ajax('GET', '/ok', full_response=True))
test_is('slow!' in r1.response.xhr.responseText, True)
test_is('ok!' in r2.response.xhr.responseText, True)
display(r1)
r2

Concurrent requests work perfectly — each got its own response with no cross-talk. But sometimes the spinner animation keeps rolling even after the cell finished exe. This suggests some problem with htmx settle.

In [ ]:
# POST 201
res = await ajax('POST', '/created', full_response=True)
test_eq(res.response.xhr.status, 201)
res

In [ ]:
# bad URL - should be {error: ...}
res = await ajax('GET', 'https://does-not-exist.invalid')
test_is('error' in res, True)
res

All cases pass cleanly. A few things to note from the output:

- **`full_response` shape differs from `swap`** — for `ajax` it's `{success: True, response: {status, responseText, pathInfo}}` rather than wrapping in `success.response`.

- **`?content=`** query param appearing in `finalRequestPath` — that's solveit's own injection, not us. Harmless but worth noting.

- **Network error** gives `'network error'` string — comes from our `.catch` fallback. Clean.

#### ajax() — findings & caveats

**What works from Python (`context` arg):**
- `None` — no swap, fire-and-forget; still blocks until request completes
- `str` selector — swaps response into target element using default `innerHTML`
- `dict` with `target` — same as above but explicit
- `dict` with `target` + `swap` — override swap style (e.g. `swap='outerHTML'`)
- `dict` with `headers` — custom request headers passed through correctly
- `dict` with `values` — extra params merged into request body/query
- `dict` with `select` — pick sub-element from response before swapping (equivalent of `hx-select`)
- `dict` with `selectOOB` — out-of-band swap additional elements from response

**`full_response=True`** adds a `response` key with:
- `xhr` — `{status, statusText, responseText, responseURL}`
- `successful` / `failed` — htmx's own assessment
- `pathInfo` — `{requestPath, finalRequestPath}`

Only meaningful for `ajax`; silently ignored on `swap` and `find`.

**Not applicable from Python:**
- `context.source` — bridge injects its own invisible detached `<div>` as source; `hx-*` attribute inheritance from a custom source element is N/A
- `context.event` — triggering event object; not serializable
- `context.handler` — custom response handler; JS function, not serializable
- `context.push` / `context.replace` — history manipulation; works but side-effects the browser URL — use deliberately

**Error handling — events monitored:**
- `htmx:afterRequest` — covers all success, 4xx/5xx, network errors, timeout, abort
- `htmx:onLoadError` — `responseHandler()` throws inside `xhr.onload`; `htmx:afterRequest` never fires in this path
- `htmx:swapError` — explicit swap error; belt-and-suspenders for `onLoadError`
- `htmx:invalidPath` — `selfRequestsOnly` violation or bad path; Promise rejects without `htmx:afterRequest`

**Timing guarantees:**
- `await ajax(...)` returns after `htmx:afterRequest` — **before** DOM settle
- If a swap target is provided, DOM may not be stable immediately after return
- Safe pattern: `await ajax(..., target)` then `await swap(...)` if settle guarantee needed, or `await find(...)` with awareness that settling may be in progress

**`?content=` query param** in `finalRequestPath` — injected by the solveit app, not the bridge; harmless.

### Method - htmx.trigger()

Triggers a given event on an element

##### Parameters

* `elt` - the element to trigger the event on
* `name` - the name of the event to trigger
* `detail` - details for the event

##### Example

```js
  // triggers the myEvent event on #tab2 with the answer 42
  htmx.trigger("#tab2", "myEvent", {answer:42});
```


In [ ]:
#| export
@delegates(_htmx_call)
async def trigger(elt, name, detail=None, **kwargs):
    "Trigger browser event `name` on `elt` with optional `detail` payload"
    return await _htmx_call('trigger', elt, name, detail or {}, **kwargs)

### Method - htmx.process()

Processes new content, enabling htmx behavior.  This can be useful if you have content that is added to the DOM
outside of the normal htmx request cycle but still want htmx attributes to work.

##### Parameters

* `elt` - element to process

##### Example

```js
  document.body.innerHTML = "<div hx-get='/example'>Get it!</div>"
  // process the newly added content
  htmx.process(document.body);
```

In [ ]:
#| export
@delegates(_htmx_call)
async def process(elt, **kwargs):
    "Process `elt` and its children, activating any htmx attributes"
    return await _htmx_call('process', elt, **kwargs)

#### trigger / process

In [ ]:
proc_out = Div('waiting...', id='proc_out')
proc_target = Div(id='proc1', hx_get='/example', hx_trigger='myProcEvent', hx_target='#proc_out')
Div(proc_out, proc_target)

In [ ]:
res = await process('#proc1')
test_is(res.success, True)
res

In [ ]:
res = await trigger('#proc1', 'myProcEvent')
test_is(res.success, True)
res

Flow is: render both divs → `process` activates the htmx attrs on `#proc1` → `trigger` fires `myProcEvent` on it → the GET to `/example` should swap the response into `#proc_out`.

Both `process` and `trigger` returned `{success: True}` — but that only tells us the JS calls didn't error. Let's verify the full round-trip actually happened by checking `#proc_out`:

In [ ]:
res = await find('#proc_out')
test_is('response!' in res.success.outerHTML, True)
res

In [ ]:
btn_wrp = Div(
    (btn := Button("Click me", hx_get=f"/example", hx_target="next div")),
    Div()
)

In [ ]:
btn_wrp

In [ ]:
await trigger(f"#{btn}", 'click')

In [ ]:
btn_wrp = Div(
    (btn := Button("Click me", hx_get=f"/not-a-route", hx_target="next div")),
    Div()
)

In [ ]:
btn_wrp

In [ ]:
await trigger(f"#{btn}", 'click')

`trigger` is genuinely fire-and-forget — `htmx.trigger()` is synchronous and just dispatches the DOM event. It returns `{success: True}` the instant the event is dispatched, with no knowledge of what happens next. Any resulting AJAX request (the GET to `/not-a-route` in the console) runs entirely asynchronously afterwards.

So `{success: True}` means "the event was dispatched without throwing", not "the resulting request succeeded". The 404 happens 200ms later and `trigger` never sees it.

This is actually correct behavior — `trigger` is the Python equivalent of clicking a button. If we need to verify what happened as a result, we follow it with `find()` to inspect the DOM, or use `ajax()` directly if want response introspection.

### blocking trigger - trigger_ex

`trigger_ex` is a blocking variant of `trigger` that waits for observable side effects. `trigger` alone is fire-and-forget; `trigger_ex` blocks until a specified htmx event fires (or an error occurs).

```python
await trigger_ex(elt, name, detail=None, wait_for='htmx:afterRequest', full_response=False)
```

It works by injecting `idx` into the trigger `detail`. htmx preserves `triggeringEvent.detail` all the way through to `afterRequest`/`afterSettle`/etc., so each listener checks `e.detail.requestConfig.triggeringEvent.detail._hxb === idx` — no DOM mutation, no headers, no query params.

`wait_for`is comma-separated list of htmx events to treat as success signals (default: `htmx:afterRequest`):
- `htmx:afterRequest` — request complete (success or HTTP error status)
- `htmx:afterSettle` — DOM fully settled after swap
- `htmx:afterSwap` — content swapped, before settle
- any custom htmx event name

`full_response=True` attaches `_ser_detail` to both success and error results:
- `{success: True, response: {xhr, successful, failed, pathInfo}}`
- `{error: 'htmx:responseError', response: {xhr: {status: 404, ...}, ...}}`

**Not applicable:**
- Pure JS/custom events that don't cause an htmx request — `wait_for` will timeout since `triggeringEvent.detail` is never checked by non-htmx handlers
- Use plain `trigger` for fire-and-forget, or follow with `find` to inspect DOM state

In [ ]:
#| export
@delegates(_htmx_call)
async def trigger_ex(elt, name, detail=None, wait_for='htmx:afterRequest', **kwargs):
    "Trigger browser event `name` on `elt`, blocking until `wait_for` event(s) fire or an error occurs"
    return await _htmx_call('trigger_ex', elt, name, detail or {}, wait_for, **kwargs)

In [ ]:
btn_wrp = Div(
    (btn := Button("Click me", hx_get=f"/example", hx_target="next div")),
    Div()
)

btn_wrp

<div id="_4KiR0vmBQ92JhXp1hXsjJA"><button hx-get="/example" hx-target="next div" id="_eOqjUaGkSTa1Qt3Kvy4omg" name="_eOqjUaGkSTa1Qt3Kvy4omg">Click me</button><div id="_y4SrTbbqRUaVcD24a_KdmA"></div></div>

In [ ]:
await trigger_ex(f"#{btn}", 'click', wait_for='htmx:afterSettle', full_response=True)

```python
{ 'response': { 'failed': False,
                'pathInfo': { 'finalRequestPath': '/example?dlg_name=vic%2Fdutil%2Fnbs%2F01_htmx_bridge&_eOqjUaGkSTa1Qt3Kvy4omg=&content=',
                              'requestPath': '/example'},
                'successful': True,
                'xhr': { 'responseText': '<p '
                                         'id="_1c15FkMCQ5mqoLq_054SFg">server '
                                         'response!</p>\n',
                         'responseURL': 'https://sweet-vision-saves-2zox4y.solve.it.com/example?dlg_name=vic%2Fdutil%2Fnbs%2F01_htmx_bridge&_eOqjUaGkSTa1Qt3Kvy4omg=&content=',
                         'status': 200,
                         'statusText': ''}},
  'success': True}
```

In [ ]:
res = await find(f"#{btn_wrp}")
test_eq(res.success.id,btn_wrp.id)
res

```python
{ 'success': { 'id': '_4KiR0vmBQ92JhXp1hXsjJA',
               'outerHTML': '<div id="_4KiR0vmBQ92JhXp1hXsjJA"><button '
                            'hx-get="/example" hx-target="next div" '
                            'id="_eOqjUaGkSTa1Qt3Kvy4omg" '
                            'name="_eOqjUaGkSTa1Qt3Kvy4omg" class="">Click '
                            'me</button><div id="_y4SrTbbqRUaVcD24a_KdmA" '
                            'class=""><p id="_1c15FkMCQ5mqoLq_054SFg">server '
                            'response!</p>\n'
                            '</div></div>',
               'tagName': 'div'}}
```

In [ ]:
btn_wrp = Div(
    (btn := Button("Click me", hx_get=f"/not-a-route", hx_target="next div")),
    Div()
)

btn_wrp

<div id="_9gkYokBcSqGEIu7ZO2FV0Q"><button hx-get="/not-a-route" hx-target="next div" id="_6Ykh1vrNTpqsBrrMuey19A" name="_6Ykh1vrNTpqsBrrMuey19A">Click me</button><div id="_cqxY77VTSFCL_swL7XL1Sg"></div></div>

In [ ]:
res = await trigger_ex(f"#{btn}", 'click')
test_is('htmx:responseError' in res.error, True)
res

```python
{'error': 'htmx:responseError'}
```

In [ ]:
res = await trigger_ex(f"#{btn}", 'click', full_response=True)
test_eq(res.response.xhr.status, 404)
res

```python
{ 'error': 'htmx:responseError',
  'response': { 'failed': True,
                'pathInfo': { 'finalRequestPath': '/not-a-route?dlg_name=vic%2Fdutil%2Fnbs%2F01_htmx_bridge&_6Ykh1vrNTpqsBrrMuey19A=&content=',
                              'requestPath': '/not-a-route'},
                'successful': False,
                'xhr': { 'responseText': '404 Not Found',
                         'responseURL': 'https://sweet-vision-saves-2zox4y.solve.it.com/not-a-route?dlg_name=vic%2Fdutil%2Fnbs%2F01_htmx_bridge&_6Ykh1vrNTpqsBrrMuey19A=&content=',
                         'status': 404,
                         'statusText': ''}}}
```

## rest of DOM wrappers

### Method - htmx.closest()

Finds the closest matching element in the given elements parentage, inclusive of the element

##### Parameters

* `elt` - the element to find the selector from
* `selector` - the selector to find

##### Example

```js
  // find the closest enclosing div of the element with the id 'demo'
  htmx.closest(htmx.find('#demo'), 'div');
```

In [ ]:
#| export
@delegates(_htmx_call)
async def closest(elt, selector, **kwargs):
    "Find closest ancestor of `elt` matching `selector`; returns dict with id, tagName, outerHTML"
    return await _htmx_call('closest', elt, selector, **kwargs)

In [ ]:
res = await closest('#p1', 'div')
test_eq(res.success.id, 'dom_root')
res

```python
{ 'success': { 'id': 'dom_root',
               'outerHTML': '<div id="dom_root"><p id="p1">paragraph one</p><p '
                            'id="p2">paragraph two</p><span id="sp1">a '
                            'span</span></div>',
               'tagName': 'div'}}
```

In [ ]:
res = await closest('#p1', 'table')
test_is('No ancestor found' in res.error, True)
res

```python
{'error': 'Error: No ancestor found: table'}
```

In [ ]:
#| export
allow('find', 'findall', 'closest')

### Method - htmx.remove()

Removes an element from the DOM

##### Parameters

* `elt` - element to remove

or

* `elt` - element to remove
* `delay` - delay (in milliseconds ) before element is removed

##### Example

```js
  // removes my-div from the DOM
  htmx.remove(htmx.find("#my-div"));

  // removes my-div from the DOM after a delay of 2 seconds
  htmx.remove(htmx.find("#my-div"), 2000);
```

In [ ]:
#| export
@delegates(_htmx_call)
async def remove(elt, delay=None, **kwargs):
    "Remove `elt` from the DOM, optionally after `delay` ms"
    return await _htmx_call('remove', elt, delay, **kwargs)

In [ ]:
target_remove = Div('I will be removed', id='rm1')
target_remove

<div id="rm1">I will be removed</div>

In [ ]:
res = await remove('#rm1')
test_eq(res, {'success': True})
res

```python
{'success': True}
```

In [ ]:
res = await remove('#does-not-exist')
test_is('Not found' in res.error, True)
res

```python
{'error': 'Error: Not found: #does-not-exist'}
```

### Method - htmx.addClass()

This method adds a class to the given element.

##### Parameters

* `elt` - the element to add the class to
* `class` - the class to add

or

* `elt` - the element to add the class to
* `class` - the class to add
* `delay` - delay (in milliseconds ) before class is added

##### Example

```js
  // add the class 'myClass' to the element with the id 'demo'
  htmx.addClass(htmx.find('#demo'), 'myClass');

  // add the class 'myClass' to the element with the id 'demo' after 1 second
  htmx.addClass(htmx.find('#demo'), 'myClass', 1000);
```

In [ ]:
#| export
@delegates(_htmx_call)
async def addClass(elt, cls, delay=None, **kwargs):
    "Add CSS `cls` to `elt`, optionally after `delay` ms"
    return await _htmx_call('addClass', elt, cls, delay, **kwargs)

### Method - htmx.removeClass()

Removes a class from the given element

##### Parameters

* `elt` - element to remove the class from
* `class` - the class to remove

or

* `elt` - element to remove the class from
* `class` - the class to remove
* `delay` - delay (in milliseconds ) before class is removed

##### Example

```js
  // removes .myClass from my-div
  htmx.removeClass(htmx.find("#my-div"), "myClass");

  // removes .myClass from my-div after 6 seconds
  htmx.removeClass(htmx.find("#my-div"), "myClass", 6000);
```

In [ ]:
#| export
@delegates(_htmx_call)
async def removeClass(elt, cls, delay=None, **kwargs):
    "Remove CSS `cls` from `elt`, optionally after `delay` ms"
    return await _htmx_call('removeClass', elt, cls, delay, **kwargs)

### Method - htmx.toggleClass()

Toggles the given class on an element

##### Parameters

* `elt` - the element to toggle the class on
* `class` - the class to toggle

##### Example

```js
  // toggles the selected class on tab2
  htmx.toggleClass(htmx.find("#tab2"), "selected");
```

In [ ]:
#| export
@delegates(_htmx_call)
async def toggleClass(elt, cls, **kwargs):
    "Toggle CSS `cls` on `elt`"
    return await _htmx_call('toggleClass', elt, cls, **kwargs)

### Method - htmx.takeClass()

Takes the given class from its siblings, so that among its siblings, only the given element will have the class.

##### Parameters

* `elt` - the element that will take the class
* `class` - the class to take

##### Example

```js
  // takes the selected class from tab2's siblings
  htmx.takeClass(htmx.find("#tab2"), "selected");
```

In [ ]:
#| export
@delegates(_htmx_call)
async def takeClass(elt, cls, **kwargs):
    "Take CSS `cls` from `elt`'s siblings so only `elt` has it"
    return await _htmx_call('takeClass', elt, cls, **kwargs)

In [ ]:
tabs = Div(
    Span('Tab1', id='tab1'),
    Span('Tab2', id='tab2'),
    Span('Tab3', id='tab3'),
    id='tabs')
tabs

<div id="tabs"><span id="tab1">Tab1</span><span id="tab2">Tab2</span><span id="tab3">Tab3</span></div>

In [ ]:
res = await addClass('#tab1', 'highlight')
test_eq(res, {'success': True})

In [ ]:
res = await removeClass('#tab1', 'highlight')
test_eq(res, {'success': True})

In [ ]:
res = await toggleClass('#tab2', 'active')
test_eq(res, {'success': True})

In [ ]:
res = await takeClass('#tab3', 'selected')
test_eq(res, {'success': True})

In [ ]:
res = await find('#tabs span:nth-child(2)')
test_is('class="active"' in res.success.outerHTML, True)
res

```python
{ 'success': { 'id': 'tab2',
               'outerHTML': '<span id="tab2" class="active">Tab2</span>',
               'tagName': 'span'}}
```

In [ ]:
res = await find('#tabs span:nth-child(3)')
test_is('class="selected"' in res.success.outerHTML, True)
res

```python
{ 'success': { 'id': 'tab3',
               'outerHTML': '<span id="tab3" class="selected">Tab3</span>',
               'tagName': 'span'}}
```

### DOM wrappers — findings & caveats

**`findAll(selector, root=None)`**
- Returns `{success: {count, items: [{id, tagName, outerHTML}]}}`
- `root` scopes the search to a subtree — matches `htmx.findAll(elt, selector)` exactly
- Empty result is `{count: 0, items: []}`, not an error
- `count` can be large (e.g. `findAll('p')` on a full solveit page returns 700+)

**`closest(elt, selector)`**
- Returns `{success: {id, tagName, outerHTML}}` of the nearest ancestor matching `selector`, inclusive of `elt`
- No match → `{error: 'No ancestor found: <selector>'}` 

**`remove(elt, delay=None)`**
- `delay` in ms, optional
- Non-existent element → `{error: "TypeError: Cannot read properties of null..."}` — htmx doesn't guard against null before calling `.parentElement`; worth wrapping with a pre-check if resilience needed

**`addClass` / `removeClass` / `toggleClass` / `takeClass`**
- All synchronous, fire-and-forget on JS side, return `{success: True}`
- `addClass`/`removeClass` accept optional `delay` ms
- `toggleClass`/`takeClass` — no delay support in htmx

**`values(elt, requestType='post')`**
- Returns htmx's resolved input values for the element
- Includes solveit-injected params (e.g. `dlg_name`) — filter these out if needed

**`parseInterval(s)`**
- Parses `'2s'` → `2000`, `'500ms'` → `500`. Caution: `'3m'` → `180000` (uses `parseFloat` fallback, not strict parsing)

**`logAll()` / `logNone()`**
- Toggle htmx console logging; useful for debugging, no meaningful return value

**`get_config(key)` / `set_config(key, value)`**
- Read/write any `htmx.config` property by name
- Mutations persist for the browser session — always restore after tests

## utility

### Method - htmx.values()

Returns the input values that would resolve for a given element via the htmx value resolution mechanism

##### Parameters

* `elt` - the element to resolve values on
* `request type` - the request type (e.g. `get` or `post`)  non-GET's will include the enclosing form of the element.
   Defaults to `post`

##### Example

```js
  // gets the values associated with this form
  var values = htmx.values(htmx.find("#myForm"));
```

In [ ]:
#| export
@delegates(_htmx_call)
async def values(elt, requestType='post', **kwargs):
    "Return input values htmx would submit for `elt` with `requestType`"
    return await _htmx_call('values', elt, requestType, **kwargs)

In [ ]:
test_form = Form(
    Input(name='username', value='alice'),
    Input(name='age', value='30', type='number'),
    id='myForm')
test_form

<form id="myForm" name="myForm"><input name="username" value="alice" id="_e-FdfiuARIGCbMPakjLMqQ"><input name="age" value="30" type="number" id="_S7G0B9CgSl6zet1kURgX3A"></form>

In [ ]:
res = await values('#myForm')
test_eq(res.success.result.age, '30')
test_eq(res.success.result.username, 'alice')
res

```python
{ 'success': { 'result': { 'age': '30',
                           'dlg_name': 'vic/dutil/nbs/01_htmx_bridge',
                           'username': 'alice'}}}
```

### Method - htmx.parseInterval()

Parses an interval string consistent with the way htmx does.  Useful for plugins that have timing-related attributes.

Caution: Accepts an int followed by either `s` or `ms`. All other values use `parseFloat`

##### Parameters

* `str` - timing string

##### Example

```js
    // returns 3000
    var milliseconds = htmx.parseInterval("3s");

    // returns 3 - Caution
    var milliseconds = htmx.parseInterval("3m");
```

In [ ]:
#| export
@delegates(_htmx_call)
async def parseInterval(s, **kwargs):
    "Parse an htmx interval string (e.g. '500ms', '2s') into milliseconds"
    return await _htmx_call('parseInterval', s, **kwargs)

### Method - htmx.logAll()

Log all htmx events, useful for debugging.

##### Example

```js
    htmx.logAll();
```

In [ ]:
#| export
@delegates(_htmx_call)
async def logAll(**kwargs):
    "Enable htmx event logging to console (useful for debugging)"
    return await _htmx_call('logAll', **kwargs)

### Method - htmx.logNone()

Log no htmx events, call this to turn off the debugger if you previously enabled it.

##### Example

```js
    htmx.logNone();
```

In [ ]:
await parseInterval('2s'), await parseInterval('500ms'), await parseInterval('3m')

({'success': {'result': 2000}},
 {'success': {'result': 500}},
 {'success': {'result': 180000}})

In [ ]:
#| export
@delegates(_htmx_call)
async def logNone(**kwargs):
    "Disable htmx event logging"
    return await _htmx_call('logNone', **kwargs)

In [ ]:
await logAll()

```python
{'success': True}
```

In [ ]:
await logNone()

```python
{'success': True}
```

## config

### Property - htmx.config

A property holding the configuration htmx uses at runtime.

Note that using a [meta tag](@/docs.md#config) is the preferred mechanism for setting these properties.

##### Properties

* `attributesToSettle:["class", "style", "width", "height"]` - array of strings: the attributes to settle during the settling phase
* `refreshOnHistoryMiss:false` - boolean: if set to `true` htmx will issue a full page refresh on history misses rather than use an AJAX request
* `defaultSettleDelay:20` - int: the default delay between completing the content swap and settling attributes
* `defaultSwapDelay:0` - int: the default delay between receiving a response from the server and doing the swap
* `defaultSwapStyle:'innerHTML'` - string: the default swap style to use if [`hx-swap`](@/attributes/hx-swap.md) is omitted
* `historyCacheSize:10` - int: the number of pages to keep in `localStorage` for history support
* `historyEnabled:true` - boolean: whether or not to use history
* `includeIndicatorStyles:true` - boolean: if true, htmx will inject a small amount of CSS into the page to make indicators invisible unless the `htmx-indicator` class is present
* `indicatorClass:'htmx-indicator'` - string: the class to place on indicators when a request is in flight
* `requestClass:'htmx-request'` - string: the class to place on triggering elements when a request is in flight
* `addedClass:'htmx-added'` - string: the class to temporarily place on elements that htmx has added to the DOM
* `settlingClass:'htmx-settling'` - string: the class to place on target elements when htmx is in the settling phase
* `swappingClass:'htmx-swapping'` - string: the class to place on target elements when htmx is in the swapping phase
* `allowEval:true` - boolean: allows the use of eval-like functionality in htmx, to enable `hx-vars`, trigger conditions & script tag evaluation.  Can be set to `false` for CSP compatibility.
* `allowScriptTags:true` - boolean: allows script tags to be evaluated in new content
* `inlineScriptNonce:''` - string: the [nonce](https://developer.mozilla.org/docs/Web/HTML/Global_attributes/nonce) to add to inline scripts
* `inlineStyleNonce:''` - string: the [nonce](https://developer.mozilla.org/docs/Web/HTML/Global_attributes/nonce) to add to inline styles
* `withCredentials:false` - boolean: allow cross-site Access-Control requests using credentials such as cookies, authorization headers or TLS client certificates
* `timeout:0` - int: the number of milliseconds a request can take before automatically being terminated
* `wsReconnectDelay:'full-jitter'` - string/function: the default implementation of `getWebSocketReconnectDelay` for reconnecting after unexpected connection loss by the event code `Abnormal Closure`, `Service Restart` or `Try Again Later`
* `wsBinaryType:'blob'` - string: the [the type of binary data](https://developer.mozilla.org/docs/Web/API/WebSocket/binaryType) being received over the WebSocket connection
* `disableSelector:"[hx-disable], [data-hx-disable]"` - array of strings: htmx will not process elements with this attribute on it or a parent
* `disableInheritance:false` - boolean: If it is set to `true`, the inheritance of attributes is completely disabled and you can explicitly specify the inheritance with the [hx-inherit](@/attributes/hx-inherit.md) attribute.
* `scrollBehavior:'instant'` - string: the scroll behavior when using the [show](@/attributes/hx-swap.md#scrolling-scroll-show) modifier with `hx-swap`. The allowed values are `instant` (scrolling should happen instantly in a single jump), `smooth` (scrolling should animate smoothly) and `auto` (scroll behavior is determined by the computed value of [scroll-behavior](https://developer.mozilla.org/en-US/docs/Web/CSS/scroll-behavior)).
* `defaultFocusScroll:false` - boolean: if the focused element should be scrolled into view, can be overridden using the [focus-scroll](@/attributes/hx-swap.md#focus-scroll) swap modifier
* `getCacheBusterParam:false` - boolean: if set to true htmx will append the target element to the `GET` request in the format `org.htmx.cache-buster=targetElementId`
* `globalViewTransitions:false` - boolean: if set to `true`, htmx will use the [View Transition](https://developer.mozilla.org/en-US/docs/Web/API/View_Transitions_API) API when swapping in new content.
* `methodsThatUseUrlParams:["get", "delete"]` - array of strings: htmx will format requests with these methods by encoding their parameters in the URL, not the request body
* `selfRequestsOnly:true` - boolean: whether to only allow AJAX requests to the same domain as the current document
* `ignoreTitle:false` - boolean: if set to `true` htmx will not update the title of the document when a `title` tag is found in new content
* `scrollIntoViewOnBoost:true` - boolean: whether or not the target of a boosted element is scrolled into the viewport. If `hx-target` is omitted on a boosted element, the target defaults to `body`, causing the page to scroll to the top.
* `triggerSpecsCache:null` - object: the cache to store evaluated trigger specifications into, improving parsing performance at the cost of more memory usage. You may define a simple object to use a never-clearing cache, or implement your own system using a [proxy object](https://developer.mozilla.org/docs/Web/JavaScript/Reference/Global_Objects/Proxy)
* `htmx.config.responseHandling:[...]` - HtmxResponseHandlingConfig[]: the default [Response Handling](@/docs.md#response-handling) behavior for response status codes can be configured here to either swap or error
* `htmx.config.allowNestedOobSwaps:true` -  boolean: whether to process OOB swaps on elements that are nested within the main response element. See [Nested OOB Swaps](@/attributes/hx-swap-oob.md#nested-oob-swaps).
* `htmx.config.historyRestoreAsHxRequest:true` -  Whether to treat history cache miss full page reload requests as a "HX-Request" by returning this response header. This should always be disabled when using HX-Request header to optionally return partial responses
* `htmx.config.reportValidityOfForms:false` -  Whether to report input validation errors to the end user and update focus to the first input that fails validation. This should always be enabled as this matches default browser form submit behaviour
##### Example

```js
  // update the history cache size to 30
  htmx.config.historyCacheSize = 30;
```

In [ ]:
#| export
@delegates(_htmx_call)
async def get_config(key, **kwargs):
    "Get an htmx config property by name"
    return await _htmx_call('get_config', key, **kwargs)

@delegates(_htmx_call)
async def set_config(key, value, **kwargs):
    "Set an htmx config property by name"
    return await _htmx_call('set_config', key, value, **kwargs)

In [ ]:
# get/set config
res = await get_config('defaultSwapStyle')
test_eq(res.success.value, 'innerHTML')
display(res)

await set_config('defaultSwapStyle', 'outerHTML')
res = await get_config('defaultSwapStyle')
test_eq(res.success.value, 'outerHTML')
display(res)

await set_config('defaultSwapStyle', 'innerHTML')  # restore

```python
{'success': {'key': 'defaultSwapStyle', 'value': 'innerHTML'}}
```

```python
{'success': {'key': 'defaultSwapStyle', 'value': 'outerHTML'}}
```

```python
{'success': True}
```

**Implemented:** `find`, `findAll`, `closest`, `remove`, `addClass`, `removeClass`, `toggleClass`, `takeClass`, `ajax`, `swap`, `process`, `trigger`, `trigger_ex`, `eval`, `values`, `parseInterval`, `logAll`, `logNone`, `get_config`/`set_config`

**Missing from JS + Python:**

- `htmx.on()` / `htmx.off()` — register/remove event listeners. Tricky since a listener is a JS function; not directly wrappable, but could support a one-shot pattern with `wait_for` built in
- `htmx.onLoad()` — callback on `htmx:load`; same issue as `on`
- `htmx.defineExtension()` / `htmx.removeExtension()` — pass JS source as string, use `eval` internally; possible but niche

**Not wrappable:**
- `htmx.createEventSource` / `htmx.createWebSocket` / `htmx.logger` — JS function properties, not serializable

**Priority candidates:** perhaps `on`/`off` but no pressure given `trigger_ex` covers the main use case. 

## non-wrapped

### Non-wrappable htmx JS API — reference

**`htmx.on(eventName, listener)` / `htmx.off(eventName, listener)`**
Not wrappable — `listener` is a JS function, not serializable. Use `trigger_ex(wait_for=...)` for the common case of observing a triggered request's outcome. For persistent listeners, use `js_eval` to inject raw JS.

**`htmx.onLoad(callback)`**
Not wrappable — callback-based. Equivalent is `trigger_ex` with `wait_for='htmx:load'`, or inject via `js_eval`.

**`htmx.defineExtension(name, ext)` / `htmx.removeExtension(name)`**
Partially wrappable — `ext` is a JS object with function properties. Could be done via `js_eval` for the full extension definition. Not implemented.

**`htmx.createEventSource` / `htmx.createWebSocket`**
JS function property assignments. Not wrappable — use `js_eval` to override if needed.

**`htmx.logger`**
JS function property. Not wrappable directly — `logAll()` / `logNone()` cover the common use case.

**Workaround for all of the above:** `js_eval('...')` can execute arbitrary JS and return any JSON-serializable result.

## one-use server

Which wrappers need a running server, which not?

**No server needed** — pure browser-side operations:
- `swap`, `remove`, `addClass`, `removeClass`, `toggleClass`, `takeClass`, `process`, `find`, `findAll`, `closest`, `values`, `parseInterval`, `logAll`, `logNone`

**Server required:**
- `ajax` — by definition makes an HTTP request
- `trigger` - sometimes

So almost everything works standalone except `ajax`. That's actually great — it means the wrapper is useful even in static/local contexts.

### ajax with local server support

In [ ]:
test_is(is_port_free(8000), False)

In [ ]:
test_is(is_port_free(8001), True)

In [ ]:
test_is(isinstance(find_server(), JupyUvi), True)

In [ ]:
app.routes

[Route(path='/{fname:path}.{ext:static}', name='static_route_exts_get', methods=['GET', 'HEAD', 'POST']),
 Route(path='/example', name='get', methods=['GET', 'HEAD']),
 Route(path='/hello', name='hello', methods=['GET', 'HEAD', 'POST']),
 Route(path='/greet', name='greet', methods=['GET', 'HEAD', 'POST']),
 Route(path='/headers-echo', name='get', methods=['GET', 'HEAD']),
 Route(path='/not-found', name='get', methods=['GET', 'HEAD']),
 Route(path='/ok', name='ok', methods=['GET', 'HEAD', 'POST']),
 Route(path='/created', name='created', methods=['GET', 'HEAD', 'POST']),
 Route(path='/slow', name='slow', methods=['GET', 'HEAD', 'POST'])]

In [ ]:
async with local_server() as srv:
    test_is(server, svr)

In [ ]:
#| export
@delegates(_htmx_call)
async def ajax(verb, path, context=None, response=None, port=8000, **kwargs):
    "Issue an htmx-style AJAX request; if `response` given, serve it locally on `port`"
    if response is None: return await _htmx_call('ajax', verb, path, context, port=port, **kwargs)
    base, _, qs = path.strip('/').partition('?')
    tmp_path = f"/_tmp_{uuid4().hex}{'/'+base if base else ''}"
    tmp_full = f"{tmp_path}?{qs}" if qs else tmp_path
    async with local_server(port=port) as srv:
        app = srv.app
        app.route(tmp_path, methods=['GET','POST','PUT','PATCH','DELETE'])(wrap_endp(response))
        route = [r for r in app.routes if r.path == tmp_path][0]
        app.routes.remove(route); app.routes.insert(0, route)
        try: return await _htmx_call('ajax', verb, tmp_full, context, True, port=port, **kwargs)
        finally: app.routes.remove(route)

### htmx.ajax() — update: local server mode

The `response` parameter is the interesting addition. If provided, it spins up (or reuses) a local server, registers a temporary route to serve that response, fires the request to it, then cleans up. This lets you test a full htmx round-trip without a pre-existing server:

`response` can be a string, FT component, sequence of str|FT, or sync/async callable receiving a Starlette `Request`.

**Usage:**
```python
# serve an FT component locally, swap into #out, block until settled
res = await ajax('GET', '/', context='#out', response=P('hello!'))

# callable response — access request params
async def handler(req): return Div(f"got: {req.query_params.get('q','')}")
res = await ajax('GET', '/?q=test', context='#out', response=handler)
```

### Test ajax with local server — existing server case

In [ ]:
target_local = Div('waiting...', id='local_out')
target_local

<div id="local_out">waiting...</div>

In [ ]:
# FT component as response
res = await ajax('GET', '/', context='#local_out', response=P('hello from local server!'))
test_eq(res, {'success': True})
res

```python
{'success': True}
```

In [ ]:
# verify temp route was cleaned up
assert not any(r.path.startswith('/_tmp/') for r in app.routes), "temp route not cleaned up!"
app.routes

[Route(path='/{fname:path}.{ext:static}', name='static_route_exts_get', methods=['GET', 'HEAD', 'POST']),
 Route(path='/example', name='get', methods=['GET', 'HEAD']),
 Route(path='/hello', name='hello', methods=['GET', 'HEAD', 'POST']),
 Route(path='/greet', name='greet', methods=['GET', 'HEAD', 'POST']),
 Route(path='/headers-echo', name='get', methods=['GET', 'HEAD']),
 Route(path='/not-found', name='get', methods=['GET', 'HEAD']),
 Route(path='/ok', name='ok', methods=['GET', 'HEAD', 'POST']),
 Route(path='/created', name='created', methods=['GET', 'HEAD', 'POST']),
 Route(path='/slow', name='slow', methods=['GET', 'HEAD', 'POST'])]

### Test `ajax` with local server — no server case

In [ ]:
server.stop()
test_is(is_port_free(8000), True)

In [ ]:
target_noserver = Div('waiting...', id='noserver_out')
target_noserver

<div id="noserver_out">waiting...</div>

In [ ]:
res = await ajax('GET', '/', context='#noserver_out', response=P('spun up on demand!'))
test_eq(res, {'success': True})
res

```python
{'success': True}
```

In [ ]:
# server should be gone after the call
test_is(is_port_free(8000), True)

In [ ]:
target_noserver = Div('waiting...', id='noserver_out2')
target_noserver

<div id="noserver_out2">waiting...</div>

In [ ]:
async def handler(req): return Div(f"got: {req.query_params.get('q','')}")

res = await ajax('GET', '/?q=test', context='#noserver_out2', response=handler)
test_eq(res, {'success': True})
res

```python
{'success': True}
```

## Use case - solveit GUI tweaks examples

> Main use case is full round-trip testing, but everythin solveit gui is HTMX so in theory we can control almost all solveit gui from python.

### vars sidebar

In [ ]:
#| export
async def toggle_vars_sidebar():
    sel = 'button[hx-get="/vars_sidebar_?oob="]'  # vars sidebar
    await trigger(sel, 'click')

In [ ]:
await toggle_vars_sidebar()

### msg collapse

In [ ]:
#| export
async def toggle_collapse(id:str=None):
    msgid = id or (await read_msg(0)).id
    sel = f'#{msgid} button[hx-post="/collapse_"]'
    await trigger(sel, 'click')

In [ ]:
await toggle_collapse((await read_msg()).id)

### clamp - the forgotten attribute

In [ ]:
#| export
async def toggle_clamp(ids:str=None, inp:bool=True, dname:str=None):
    id = list(map(str.strip, ids.split(',')))[0] if ids else (await read_msg(0)).id
    dname = dname or find_dname().strip('/')
    d = {"values": {"is_input": int(inp), "dlg_name": dname, "id_": id, "ids": ids or id}, "swap": "none"}
    await ajax('POST', '/clamp_', d, full_response=True)

In [ ]:
src = pydoc.render_doc(sys.modules['dutil.hx_bridge'], renderer=pydoc.plaintext)
msgid = await link_msg(src, msg_type='raw')
await toggle_clamp(msgid)

### select message

In [ ]:
#| export
async def select_msg(id:str=None):
    id = id or (await read_msg(0)).id
    await js_eval("selectMsg($('#%s'),{centered: true})" % id)

In [ ]:
await select_msg((await read_msg()).id)

### get dialog name

In [ ]:
async def get_dialog_name():
    res = await js_eval('dialogName()')
    return res

### dialog info

In [ ]:
async def dialog_info():
    res = await js_eval('dialogName()')
    return res

# export -

In [ ]:
await ctxusage()

43075

In [ ]:
from dutil.flakes import show_flakes
await show_flakes()

<div class="prose">

No warnings to report

</div>

In [ ]:
# #|hide
# #|eval: false
# from dutil.core import dlg_export
# dlg_export()